# Import Libraries and Set Up Basic Functions

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.models import load_model
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import concurrent.futures

# Function to create sequences
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data.iloc[i:(i + seq_length)].values
        y = data.iloc[i + seq_length]['Close']
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

# Feature Engineering: Add technical indicators
def add_indicators(df):
    df['RSI'] = compute_RSI(df['Close'])
    df['MACD'], df['MACD_Signal'] = compute_MACD(df['Close'])
    df.dropna(inplace=True)
    return df

# Compute RSI
def compute_RSI(series, period=14):
    delta = series.diff(1)
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

# Compute MACD
def compute_MACD(series, short_period=12, long_period=26, signal_period=9):
    short_ema = series.ewm(span=short_period, adjust=False).mean()
    long_ema = series.ewm(span=long_period, adjust=False).mean()
    macd = short_ema - long_ema
    macd_signal = macd.ewm(span=signal_period, adjust=False).mean()
    return macd, macd_signal

# Process the data
def process_data(df, seq_length, stock_code):
    df = add_indicators(df)
    scaler = MinMaxScaler(feature_range=(0, 1))
    
    # Scale features and target separately
    feature_columns = ['Open', 'High', 'Low', 'Close', 'Volume', 'RSI', 'MACD']
    scaled_features = scaler.fit_transform(df[feature_columns])
    
    df_scaled = pd.DataFrame(scaled_features, columns=feature_columns, index=df.index)
    
    X, y = create_sequences(df_scaled, seq_length)
    
    # Split data
    train_size = len(X) - int(24 * 60 / 5)  # Use last day for testing
    X_train, y_train = X[:train_size], y[:train_size]
    X_test, y_test = X[train_size:], y[train_size:]

    # Save the scaler
    joblib.dump(scaler, f'scaler_{stock_code}.pkl')
    
    return X_train, y_train, X_test, y_test, scaler


# Build the BiLSTM model
def build_bilstm_model(input_shape):
    model = Sequential()
    model.add(Bidirectional(LSTM(100, return_sequences=True), input_shape=input_shape))
    model.add(Bidirectional(LSTM(100, return_sequences=False)))
    model.add(Dropout(0.3))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(1, activation='linear'))

    model.compile(optimizer='adam', loss=MeanSquaredError(), metrics=[RootMeanSquaredError()])
    return model

# Train the model
def train_model(X_train, y_train, X_test, y_test, seq_length, stock_code):
    model = build_bilstm_model((seq_length, X_train.shape[2]))
    cp = ModelCheckpoint(f'model_{stock_code}.keras', save_best_only=True)
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=20, batch_size=16, callbacks=[cp])
    return model, history

# Evaluate and plot results
def evaluate_and_plot_with_metrics(stock_code, model, X_train, y_train, X_test, y_test, scaler):
    train_predictions = model.predict(X_train)
    test_predictions = model.predict(X_test)
    
    # Construct arrays with the same shape expected by the scaler
    train_features_dummy = np.zeros((train_predictions.shape[0], 7))
    test_features_dummy = np.zeros((test_predictions.shape[0], 7))
    
    # Place the predictions in the 'Close' position
    train_features_dummy[:, 3] = train_predictions[:, 0]  # Assuming 'Close' is the 4th column
    test_features_dummy[:, 3] = test_predictions[:, 0]    # Assuming 'Close' is the 4th column
    
    # Inverse transform the predictions
    train_predictions_rescaled = scaler.inverse_transform(train_features_dummy)[:, 3]
    test_predictions_rescaled = scaler.inverse_transform(test_features_dummy)[:, 3]
    
    # Inverse transform the actual values
    y_train_dummy = np.zeros((y_train.shape[0], 7))
    y_test_dummy = np.zeros((y_test.shape[0], 7))
    y_train_dummy[:, 3] = y_train
    y_test_dummy[:, 3] = y_test
    
    y_train_rescaled = scaler.inverse_transform(y_train_dummy)[:, 3]
    y_test_rescaled = scaler.inverse_transform(y_test_dummy)[:, 3]
    
    # Calculate metrics
    train_rmse = np.sqrt(mean_squared_error(y_train_rescaled, train_predictions_rescaled))
    test_rmse = np.sqrt(mean_squared_error(y_test_rescaled, test_predictions_rescaled))
    train_r2 = r2_score(y_train_rescaled, train_predictions_rescaled)
    test_r2 = r2_score(y_test_rescaled, test_predictions_rescaled)
    train_mape = np.mean(np.abs((y_train_rescaled - train_predictions_rescaled) / y_train_rescaled)) * 100
    test_mape = np.mean(np.abs((y_test_rescaled - test_predictions_rescaled) / y_test_rescaled)) * 100
    
    # Plotting results
    plt.figure(figsize=(14, 5))
    plt.plot(y_train_rescaled, color='blue', label='Actual Train')
    plt.plot(np.arange(len(y_train_rescaled), len(y_train_rescaled) + len(y_test_rescaled)), y_test_rescaled, color='green', label='Actual Test')
    plt.plot(train_predictions_rescaled, color='orange', label='Predicted Train')
    plt.plot(np.arange(len(y_train_rescaled), len(y_train_rescaled) + len(y_test_rescaled)), test_predictions_rescaled, color='red', label='Predicted Test')
    
    # Add text labels for metrics
    plt.text(0.02, 0.75, f'Train RMSE: {train_rmse:.4f}\nTest RMSE: {test_rmse:.4f}', transform=plt.gca().transAxes, fontsize=10, verticalalignment='top')
    plt.text(0.02, 0.65, f'Train R2: {train_r2:.4f}\nTest R2: {test_r2:.4f}', transform=plt.gca().transAxes, fontsize=10, verticalalignment='top')
    plt.text(0.02, 0.55, f'Train MAPE: {train_mape:.2f}%\nTest MAPE: {test_mape:.2f}%', transform=plt.gca().transAxes, fontsize=10, verticalalignment='top')
    
    plt.xlabel('Time')
    plt.ylabel('Close Price')
    plt.title(f'Stock Price Close Prediction for {stock_code}')
    plt.legend()
    plt.show()

    return {
        'stock': stock_code,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'train_mape': train_mape,
        'test_mape': test_mape,
        'y_train_rescaled': y_train_rescaled,
        'y_test_rescaled': y_test_rescaled,
        'train_predictions_rescaled': train_predictions_rescaled,
        'test_predictions_rescaled': test_predictions_rescaled
    }

# Training

In [ ]:
# Fetch data from Yahoo Finance
def fetch_data(stock_code, interval='5m', period='1mo'):
    ticker = yf.Ticker(stock_code)
    data = ticker.history(interval=interval, period=period)
    return data

# List of top 10 DOW JONES stocks
dow_jones_stocks = ['AAPL', 'MSFT', 'JNJ', 'V', 'WMT', 'JPM', 'PG', 'UNH', 'DIS', 'HD']

# Train and evaluate models in parallel
results = []

def train_and_evaluate_stock(stock_code):
    try:
        stock_data = fetch_data(stock_code, interval='5m', period='1mo')
        seq_length = 4
        df_selected = pd.DataFrame(stock_data)
        X_train, y_train, X_test, y_test, scaler = process_data(df_selected, seq_length, stock_code)
        model, history = train_model(X_train, y_train, X_test, y_test, seq_length, stock_code)
        metrics = evaluate_and_plot_with_metrics(stock_code, model, X_train, y_train, X_test, y_test, scaler)
        return metrics
    except Exception as e:
        print(f"Error processing stock {stock_code}: {e}")
        return None

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(train_and_evaluate_stock, stock) for stock in dow_jones_stocks]
    for future in concurrent.futures.as_completed(futures):
        result = future.result()
        if result:
            results.append(result)

# Select top 5 stocks with best MAPE
top_5_stocks = sorted(results, key=lambda x: x['test_rmse'])[:5]

# Print the results for the top 5 stocks and plot their results
for stock in top_5_stocks:
    print(f"Stock: {stock['stock']}")
    print(f"Train RMSE: {stock['train_rmse']}")
    print(f"Test RMSE: {stock['test_rmse']}")
    print(f"Train R2: {stock['train_r2']}")
    print(f"Test R2: {stock['test_r2']}")
    print(f"Train MAPE: {stock['train_mape']}")
    print(f"Test MAPE: {stock['test_mape']}")
    print("\n")
    
    # Plotting the top 5 stocks
    plt.figure(figsize=(14, 5))
    plt.plot(stock['y_train_rescaled'], color='blue', label='Actual Train')
    plt.plot(np.arange(len(stock['y_train_rescaled']), len(stock['y_train_rescaled']) + len(stock['y_test_rescaled'])), stock['y_test_rescaled'], color='green', label='Actual Test')
    plt.plot(stock['train_predictions_rescaled'], color='orange', label='Predicted Train')
    plt.plot(np.arange(len(stock['y_train_rescaled']), len(stock['y_train_rescaled']) + len(stock['y_test_rescaled'])), stock['test_predictions_rescaled'], color='red', label='Predicted Test')
    
    # Add text labels for metrics
    plt.text(0.02, 0.75, f'Train RMSE: {stock['train_rmse']:.4f}\nTest RMSE: {stock['test_rmse']:.4f}', transform=plt.gca().transAxes, fontsize=10, verticalalignment='top')
    plt.text(0.02, 0.65, f'Train R2: {stock['train_r2']:.4f}\nTest R2: {stock['test_r2']:.4f}', transform=plt.gca().transAxes, fontsize=10, verticalalignment='top')
    plt.text(0.02, 0.55, f'Train MAPE: {stock['train_mape']:.2f}%\nTest MAPE: {stock['test_mape']:.2f}%', transform=plt.gca().transAxes, fontsize=10, verticalalignment='top')
    
    plt.xlabel('Time')
    plt.ylabel('Close Price')
    plt.title(f'Stock Price Close Prediction for {stock['stock']}')
    plt.legend()
    plt.show()


In [5]:
list_stocks = []
for s in top_5_stocks:
    print(s['stock'], s['test_rmse'])
    list_stocks.append(s['stock'])

list_stocks

WMT 0.09195445620493638
JNJ 0.19572146814916327
PG 0.26070942989645896
DIS 0.38680111477900725
HD 0.6413807514117332


['WMT', 'JNJ', 'PG', 'DIS', 'HD']

# Backtest Trading
Given the 5 best performing models, try backtest trading for an unseen day and see the most profitable stock for today.

In [6]:
from datetime import datetime, timedelta

top_5_dow_jones_stocks = list_stocks
start_backtest = (datetime.today() - timedelta(days=2)).strftime('%Y-%m-%d')
end_backtest = (datetime.today() - timedelta(days=1)).strftime('%Y-%m-%d')
start_backtest, end_backtest

('2024-07-02', '2024-07-03')

In [8]:
SEQ_LENGTH = 4
MAX_POSITION_SIZE = 1000  # Maximum number of shares to buy/sell at once
TRANSACTION_COST = 0.001  # 0.1% transaction cost
MARGIN_REQUIREMENT = 0.5  # 50% margin requirement for short selling
SLIPPAGE = 0.001  # 0.1% slippage

In [10]:
# Function to preprocess data for prediction
def preprocess_data_for_prediction(data, scaler):
    # Add missing columns with default values
    required_columns = ['Open', 'High', 'Low', 'Close', 'Volume', 'RSI', 'MACD']
    for col in required_columns:
        if (col not in data.columns) or (data[col].isnull().any()):
            data[col] = 0
    data_scaled = scaler.transform(data[required_columns])
    return data_scaled

# Function to predict the next closing price
def predict_next_closing_price(model, scaler, data_for_prediction):
    # Preprocess data for prediction
    data_for_prediction = preprocess_data_for_prediction(data_for_prediction, scaler)
    data_for_prediction = np.array([data_for_prediction])  # Reshape to 3D array
    
    # Predict with the model
    predicted_price_scaled = model.predict(data_for_prediction)
    
    # Construct a dummy array to mimic the original feature structure
    features_dummy = np.zeros((predicted_price_scaled.shape[0], 7))
    features_dummy[:, 3] = predicted_price_scaled[:, 0]  # Assuming 'Close' is the 4th column
    
    # Inverse transform to get the actual predicted price
    predicted_price = scaler.inverse_transform(features_dummy)[:, 3]
    
    # Extract the scalar value from the array
    predicted_price = predicted_price[0]
    
    return predicted_price

# Function to execute trading strategy
def execute_trading(model, scaler, historical_data):
    initial_cash = 10000  # Example starting cash
    cash = initial_cash
    shares = 0
    short_shares = 0
    transactions = []
    portfolio_value = [initial_cash]

    if historical_data.empty or len(historical_data) < SEQ_LENGTH:
        raise ValueError("Not enough data to perform backtesting.")

    for i in range(len(historical_data) - SEQ_LENGTH):
        current_data = historical_data.iloc[i:i+SEQ_LENGTH]
        data_for_prediction = current_data[['Open', 'High', 'Low', 'Close', 'Volume', 'RSI', 'MACD']]
        
        if len(data_for_prediction) == SEQ_LENGTH:
            predicted_price = predict_next_closing_price(model, scaler, data_for_prediction)
            current_price = historical_data['Close'].iloc[i + SEQ_LENGTH]
            
            # Adjust for transaction costs and slippage
            buy_price = current_price * (1 + TRANSACTION_COST + SLIPPAGE)
            sell_price = current_price * (1 - TRANSACTION_COST - SLIPPAGE)
            
            # Trading logic
            if predicted_price > buy_price and cash > buy_price:
                shares_to_buy = min(cash // buy_price, MAX_POSITION_SIZE)
                if shares_to_buy > 0:
                    cash -= shares_to_buy * buy_price
                    shares += shares_to_buy
                    transactions.append(('buy', shares_to_buy, buy_price))
            
            elif predicted_price < sell_price and shares > 0:
                shares_to_sell = min(shares, MAX_POSITION_SIZE)
                if shares_to_sell > 0:
                    cash += shares_to_sell * sell_price
                    shares -= shares_to_sell
                    transactions.append(('sell', shares_to_sell, sell_price))
            
            # Short selling logic
            elif predicted_price < sell_price and cash > sell_price * MARGIN_REQUIREMENT:
                shares_to_short = min(cash // (sell_price * MARGIN_REQUIREMENT), MAX_POSITION_SIZE)
                if shares_to_short > 0:
                    cash += shares_to_short * sell_price
                    short_shares += shares_to_short
                    transactions.append(('short_sell', shares_to_short, sell_price))
            
            elif predicted_price > buy_price and short_shares > 0:
                shares_to_cover = min(short_shares, MAX_POSITION_SIZE)
                if shares_to_cover > 0:
                    cash -= shares_to_cover * buy_price
                    short_shares -= shares_to_cover
                    transactions.append(('cover_short', shares_to_cover, buy_price))
        
        # Calculate current portfolio value and append to the list
        current_portfolio_value = cash + shares * current_price - short_shares * current_price
        portfolio_value.append(current_portfolio_value)
        
        # Stop trading if the return exceeds 100%
        if (current_portfolio_value - initial_cash) / initial_cash >= 1.0:
            print(f"Stopping trading as 100% return achieved: {current_portfolio_value}")
            break

    final_portfolio_value = cash + shares * historical_data['Close'].iloc[-1] - short_shares * historical_data['Close'].iloc[-1]
    return initial_cash, final_portfolio_value, transactions, portfolio_value

# Function to backtest trading strategy
def backtest_trading_strategy(stock_code, model_path, scaler_path, start_date='2024-07-02', end_date='2024-07-02'):
    model = load_model(model_path)
    scaler = joblib.load(scaler_path)
    
    # Fetch historical data for the backtest period
    historical_data = fetch_data(stock_code, interval='1m', period='1d')
    if historical_data.empty:
        print(f"Skipping backtesting for {stock_code} due to data fetching issues.")
        return None
    
    # Execute trading strategy
    initial_cash, final_portfolio_value, transactions, portfolio_value = execute_trading(model, scaler, historical_data)
    
    # Calculate returns
    returns = (final_portfolio_value - initial_cash) / initial_cash
    
    # Print results
    print(f"Initial Cash: {initial_cash}")
    print(f"Final Portfolio Value for {stock_code}: {final_portfolio_value}")
    print(f"Returns for {stock_code}: {returns * 100:.2f}%")
    print(f"Transactions for {stock_code}: {transactions}")
    
    # Prepare data for plotting
    predicted_prices = []
    actual_prices = []
    timestamps = []
    for i in range(len(historical_data) - SEQ_LENGTH):
        current_data = historical_data.iloc[i:i+SEQ_LENGTH]
        data_for_prediction = current_data[['Open', 'High', 'Low', 'Close', 'Volume', 'RSI', 'MACD']]
        
        if len(data_for_prediction) == SEQ_LENGTH:
            predicted_price = predict_next_closing_price(model, scaler, data_for_prediction)
            actual_price = historical_data['Close'].iloc[i + SEQ_LENGTH]
            timestamp = historical_data.index[i + SEQ_LENGTH]
            
            predicted_prices.append(predicted_price)
            actual_prices.append(actual_price)
            timestamps.append(timestamp)
    
    # Plotting predicted vs actual prices
    plt.figure(figsize=(14, 7))
    plt.plot(timestamps, actual_prices, label='Actual Close Price', marker='o')
    plt.plot(timestamps, predicted_prices, label='Predicted Close Price', marker='x')
    plt.title(f'Actual vs Predicted Closing Prices for {stock_code}')
    plt.xlabel('Timestamp')
    plt.ylabel('Price')
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # Plotting portfolio value over time
    plt.figure(figsize=(14, 7))
    plt.plot(portfolio_value, label='Portfolio Value', marker='o')
    plt.title('Portfolio Value Over Time')
    plt.xlabel('Time Step')
    plt.ylabel('Portfolio Value')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return {
        'stock_code': stock_code,
        'initial_cash': initial_cash,
        'final_portfolio_value': final_portfolio_value,
        'returns': returns,
        'transactions': transactions,
        'portfolio_value': portfolio_value
    }

# Load the top 5 Dow Jones stocks based on model test RMSE
top_5_dow_jones_stocks = ['AAPL', 'MSFT', 'JPM', 'V', 'PG']  # Example list of stocks

# Define the backtest period
start_backtest = '2024-07-02'
end_backtest = '2024-07-02'

# Backtest trading strategy for each of the top 5 stocks in parallel
results = []
with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(backtest_trading_strategy, stock, f'model_{stock}.keras', f'scaler_{stock}.pkl', start_backtest, end_backtest) for stock in top_5_dow_jones_stocks]
    for future in concurrent.futures.as_completed(futures):
        result = future.result()
        if result:
            results.append(result)

# Sort results by returns and report the top 3
top_3_results = sorted(results, key=lambda x: x['returns'], reverse=True)[:3]

# Print overall results
for result in top_3_results:
    print(f"Stock: {result['stock_code']}")
    print(f"Initial Cash: {result['initial_cash']}")
    print(f"Final Portfolio Value: {result['final_portfolio_value']}")
    print(f"Returns: {result['returns'] * 100:.2f}%")
    print(f"Transactions: {result['transactions']}")
    print("\n")


KeyError: "['RSI', 'MACD'] not in index"

In [ ]:
# Print overall results
for result in top_3_results:
    print(f"Stock: {result['stock_code']}")
    print(f"Initial Cash: {result['initial_cash']}")
    print(f"Final Portfolio Value: {result['final_portfolio_value']}")
    print(f"Returns: {result['returns'] * 100:.2f}%")
    print(f"Transactions: {result['transactions']}")
    print("\n")


In [ ]:
for s in top_3_results:
    top_stock = s['stock_code']
    break

top_stock

# Retrain the model for the best performing stock with data up until today

In [ ]:
from datetime import datetime
train_start = '2024-06-01'
train_end = (datetime.today() - timedelta(days=2)).strftime('%Y-%m-%d') # Day before Yesterday
# train_end = (datetime.today() - timedelta(days=1)).strftime('%Y-%m-%d') # Yesterday
train_start, train_end

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.losses import MeanSquaredError
import joblib

# Function to create sequences
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data.iloc[i:(i + seq_length)].values
        y = data.iloc[i + seq_length]['Close']
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

# Fetch data from Yahoo Finance
def fetch_data(stock_code, interval='5m', start='2024-06-01', end='2024-06-30'):
    ticker = yf.Ticker(stock_code)
    data = ticker.history(interval=interval, start=start, end=end)
    if data.empty:
        raise ValueError(f"No data found for {stock_code} with the given parameters.")
    return data

# Feature Engineering: Add technical indicators
def add_indicators(df):
    df['RSI'] = compute_RSI(df['Close'])
    df['MACD'], df['MACD_Signal'] = compute_MACD(df['Close'])
    df.dropna(inplace=True)
    return df

# Compute RSI
def compute_RSI(series, period=14):
    delta = series.diff(1)
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

# Compute MACD
def compute_MACD(series, short_period=12, long_period=26, signal_period=9):
    short_ema = series.ewm(span=short_period, adjust=False).mean()
    long_ema = series.ewm(span=long_period, adjust=False).mean()
    macd = short_ema - long_ema
    macd_signal = macd.ewm(span=signal_period, adjust=False).mean()
    return macd, macd_signal

# Process the data
def process_data(df, seq_length, stock_code):
    df = add_indicators(df)
    scaler = MinMaxScaler(feature_range=(0, 1))
    
    # Scale features and target separately
    feature_columns = ['Open', 'High', 'Low', 'Close', 'Volume', 'RSI', 'MACD']
    scaled_features = scaler.fit_transform(df[feature_columns])
    
    df_scaled = pd.DataFrame(scaled_features, columns=feature_columns, index=df.index)
    
    X, y = create_sequences(df_scaled, seq_length)

    # Save the scaler
    joblib.dump(scaler, f'scaler_{stock_code}.pkl')
    
    return X, y, scaler

# Build the BiLSTM model
def build_bilstm_model(input_shape):
    model = Sequential()
    model.add(Bidirectional(LSTM(100, return_sequences=True), input_shape=input_shape))
    model.add(Bidirectional(LSTM(100, return_sequences=False)))
    model.add(Dropout(0.3))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(1, activation='linear'))

    model.compile(optimizer='adam', loss=MeanSquaredError(), metrics=[RootMeanSquaredError()])
    return model

# Train the model
def train_model(X, y, seq_length, stock_code):
    model = build_bilstm_model((seq_length, X.shape[2]))
    # Ensure the directory exists
    if not os.path.exists('models'):
        os.makedirs('models')
    cp = ModelCheckpoint(f'model_{stock_code}_latest.keras', save_best_only=True, monitor='val_loss')
    history = model.fit(X, y, epochs=20, batch_size=16, validation_split=0.2, callbacks=[cp])
    return model, history

# Stock code to train
stock_code = 'PG'  # Replace with the desired stock code

# Fetch data
stock_data = fetch_data(stock_code, interval='5m', start=train_start, end=train_end)

# Process data
seq_length = 4
df_selected = pd.DataFrame(stock_data)
X, y, scaler = process_data(df_selected, seq_length, stock_code)

# Train model
model, history = train_model(X, y, seq_length, stock_code)

print(f"Model for {stock_code} has been trained and saved as 'model_{stock_code}_latest.keras'.")



# Optional Back-Test on Latest Day

# Actual Trading Simulation
Trade the most promising stock throughout the night and come out with a returns

In [ ]:
import yfinance as yf

# Define the stock ticker symbol
stock_code = top_stock

# Fetch the latest data for the stock
ticker = yf.Ticker(stock_code)
latest_data = ticker.history(period='1d', interval='1m')

# Get the latest price
if not latest_data.empty:
    latest_price = latest_data['Close'].iloc[-1]
    print(f"The latest price of {stock_code} is: {latest_price}")
else:
    print(f"No data available for {stock_code}")


In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
import joblib
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt
import time
from datetime import datetime, timedelta

SEQ_LENGTH = 4
MAX_POSITION_SIZE = 1000  # Maximum number of shares to buy/sell at once
TRANSACTION_COST = 0.001  # 0.1% transaction cost
MARGIN_REQUIREMENT = 0.5  # 50% margin requirement for short selling
SLIPPAGE = 0.001  # 0.1% slippage
MARKET_CLOSE_HOUR = 16  # Market close hour in 24-hour format (e.g., 16 for 4 PM)

# Function to fetch data from Yahoo Finance
def fetch_data(stock_code, interval='5m'):
    try:
        data = yf.download(stock_code, interval=interval, period='5d')
        if data.empty:
            raise ValueError(f"No data found for {stock_code} with the given parameters.")
        data['RSI'] = compute_rsi(data['Close'])
        data['MACD'], data['MACD_Signal'] = compute_macd(data['Close'])
        return data
    except Exception as e:
        print(f"Error fetching data for {stock_code}: {e}")
        return pd.DataFrame()

# Function to fetch the latest price from Yahoo Finance
def fetch_latest_price(stock_code):
    try:
        ticker = yf.Ticker(stock_code)
        data = ticker.history(period='1d', interval='1m')
        if data.empty:
            raise ValueError(f"No data found for {stock_code} with the given parameters.")
        latest_row = data.iloc[-1]
        return latest_row.name, latest_row['Open'], latest_row['High'], latest_row['Low'], latest_row['Close'], latest_row['Volume']
    except Exception as e:
        print(f"Error fetching latest price for {stock_code}: {e}")
        return None, None, None, None, None, None

# Function to compute RSI
def compute_rsi(data, window=14):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

# Function to compute MACD
def compute_macd(data, slow=26, fast=12, signal=9):
    exp1 = data.ewm(span=fast, adjust=False).mean()
    exp2 = data.ewm(span=slow, adjust=False).mean()
    macd = exp1 - exp2
    signal_line = macd.ewm(span=signal, adjust=False).mean()
    return macd, signal_line

# Function to preprocess data for prediction
def preprocess_data_for_prediction(data, scaler):
    # Add missing columns with default values
    required_columns = ['Open', 'High', 'Low', 'Close', 'Volume', 'RSI', 'MACD']
    for col in required_columns:
        if (col not in data.columns) or (data[col].isnull().any()):
            data[col] = 0
    data_scaled = scaler.transform(data[required_columns])
    return data_scaled

# Function to predict the next closing price
def predict_next_closing_price(model, scaler, data_for_prediction):
    # Preprocess data for prediction
    data_for_prediction = preprocess_data_for_prediction(data_for_prediction, scaler)
    data_for_prediction = np.array([data_for_prediction])  # Reshape to 3D array
    
    # Predict with the model
    predicted_price_scaled = model.predict(data_for_prediction)
    
    # Construct a dummy array to mimic the original feature structure
    features_dummy = np.zeros((predicted_price_scaled.shape[0], 7))
    features_dummy[:, 3] = predicted_price_scaled[:, 0]  # Assuming 'Close' is the 4th column
    
    # Inverse transform to get the actual predicted price
    predicted_price = scaler.inverse_transform(features_dummy)[:, 3]
    
    # Extract the scalar value from the array
    predicted_price = predicted_price[0]
    
    return predicted_price

# Function to execute trading strategy
def execute_trading(model, scaler, historical_data, latest_price, cash, shares, short_shares, transactions):
    if historical_data.empty or len(historical_data) < SEQ_LENGTH:
        raise ValueError("Not enough data to perform trading.")

    current_data = historical_data.iloc[-SEQ_LENGTH:]
    data_for_prediction = current_data[['Open', 'High', 'Low', 'Close', 'Volume', 'RSI', 'MACD']]
    
    if len(data_for_prediction) == SEQ_LENGTH:
        predicted_price = predict_next_closing_price(model, scaler, data_for_prediction)
        current_price = latest_price
        
        # Adjust for transaction costs and slippage
        buy_price = current_price * (1 + TRANSACTION_COST + SLIPPAGE)
        sell_price = current_price * (1 - TRANSACTION_COST - SLIPPAGE)
        
        # Trading logic
        if predicted_price > buy_price and cash > buy_price:
            shares_to_buy = min(cash // buy_price, MAX_POSITION_SIZE)
            if shares_to_buy > 0:
                cash -= shares_to_buy * buy_price
                shares += shares_to_buy
                transactions.append(('buy', shares_to_buy, buy_price))
        
        elif predicted_price < sell_price and shares > 0:
            shares_to_sell = min(shares, MAX_POSITION_SIZE)
            if shares_to_sell > 0:
                cash += shares_to_sell * sell_price
                shares -= shares_to_sell
                transactions.append(('sell', shares_to_sell, sell_price))
        
        # Short selling logic
        elif predicted_price < sell_price and cash > sell_price * MARGIN_REQUIREMENT:
            shares_to_short = min(cash // (sell_price * MARGIN_REQUIREMENT), MAX_POSITION_SIZE)
            if shares_to_short > 0:
                cash += shares_to_short * sell_price
                short_shares += shares_to_short
                transactions.append(('short_sell', shares_to_short, sell_price))
        
        elif predicted_price > buy_price and short_shares > 0:
            shares_to_cover = min(short_shares, MAX_POSITION_SIZE)
            if shares_to_cover > 0:
                cash -= shares_to_cover * buy_price
                short_shares -= shares_to_cover
                transactions.append(('cover_short', shares_to_cover, buy_price))
        
        # Calculate current portfolio value and append to the list
        current_portfolio_value = cash + shares * current_price - short_shares * current_price
        return predicted_price, cash, shares, short_shares, transactions, current_portfolio_value

# Function to continuously fetch the latest price and make trading decisions
def live_trading(stock_code, model_path, scaler_path, historical_data):
    model = load_model(model_path)
    scaler = joblib.load(scaler_path)
    
    # Initial cash and positions
    initial_cash = 10000  # Example starting cash
    cash = initial_cash
    shares = 0
    short_shares = 0
    transactions = []
    portfolio_values = [initial_cash]
    last_valid_close = None
    
    # Loop until the market closes
    while datetime.now().hour < MARKET_CLOSE_HOUR:
        latest_time, latest_open, latest_high, latest_low, latest_close, latest_volume = fetch_latest_price(stock_code)
        if latest_close is not None:
            last_valid_close = latest_close
            new_row = pd.DataFrame({
                'Open': [latest_open],
                'High': [latest_high],
                'Low': [latest_low],
                'Close': [latest_close],
                'Volume': [latest_volume]
            }, index=[latest_time])
            
            # Update historical data
            historical_data = pd.concat([historical_data, new_row])
            historical_data['RSI'] = compute_rsi(historical_data['Close'])
            historical_data['MACD'], historical_data['MACD_Signal'] = compute_macd(historical_data['Close'])
            
            # Execute trading
            predicted_price, cash, shares, short_shares, transactions, portfolio_value = execute_trading(
                model, scaler, historical_data, latest_close, cash, shares, short_shares, transactions)
            portfolio_values.append(portfolio_value)
            
            print(f"Latest Price: {latest_close}, Next Predicted Price: {predicted_price}, Portfolio Value: {portfolio_value}")
        
        # Wait for a minute before fetching the latest price again
        time.sleep(60)
    
    # Use the last valid close price if the latest fetch fails
    final_close_price = last_valid_close if last_valid_close is not None else latest_close
    final_portfolio_value = cash + shares * final_close_price - short_shares * final_close_price
    returns = (final_portfolio_value - initial_cash) / initial_cash
    
    print(f"Final Portfolio Value: {final_portfolio_value}")
    print(f"Returns: {returns * 100:.2f}%")
    print(f"Transactions: {transactions}")

    # Plotting portfolio value over time
    plt.figure(figsize=(14, 7))
    plt.plot(portfolio_values, label='Portfolio Value', marker='o')
    plt.title('Portfolio Value Over Time')
    plt.xlabel('Time Step')
    plt.ylabel('Portfolio Value')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return {
        'initial_cash': initial_cash,
        'final_portfolio_value': final_portfolio_value,
        'returns': returns,
        'transactions': transactions,
        'portfolio_values': portfolio_values
    }

# Define the stock code and model details
stock_code = top_stock
model_path = f'model_{stock_code}_latest.keras'
scaler_path = f'scaler_{stock_code}.pkl'
start_backtest = '2023-06-23'
end_backtest = '2024-07-02'

# Fetch historical data
historical_data = fetch_data(stock_code, interval='5m')

# Execute live trading
result = live_trading(stock_code, model_path, scaler_path, historical_data)

# Print the results
if result:
    print(f"Initial Cash: {result['initial_cash']}")
    print(f"Final Portfolio Value: {result['final_portfolio_value']}")
    print(f"Returns: {result['returns'] * 100:.2f}%")
    print(f"Transactions: {result['transactions']}")
